# Link and GPS quality along the track -- rekon10

**What this is.** For one FC log, every link and positioning quality figure the vehicle and
the ground recorded, placed on the track at the EKF's position (`POS`) and shown as map
layers. It presents what each stream reports and does not grade it.

**Layers, and where each comes from:**

| layer | source | what it is |
|---|---|---|
| GPS fix, sats, HDop | `GPS` | the F9P's fix type and geometry |
| HAcc, VAcc | `GPA` | the F9P's own accuracy estimates, m |
| jamInd, noisePerMS, agcCnt | `UBX1` | the F9P's view of its RF environment (MON-HW) |
| uplink LQ, uplink RSSI (radio 1), uplink SNR, receiver MAVLink buffer | `RAD` | the ELRS receiver hearing the ground. Decoded per the receiver source; see `vehicle_health.elrs_radio` |
| ELRS link MAVLink rx/tx, drops | `MAV` chan 2 (= `MAV3`, SERIAL6) | packets the FC received and sent on the ground link, per second |
| coordinator wlan0 rx/tx, drops, errors | the coordinator session's collectd `interface-wlan0` | bytes per second on the coordinator's wifi |
| ground backpack MAVLink packets down | `ground/backpack-metrics-*.json` | packets the ground backpack received from the TX module, per second |

**Not recorded, so not shown.** The coordinator's wifi RSSI (collectd has no wireless plugin
configured; only interface counters). The ELRS receiver's second radio, which radio is active,
the downlink (ground receiving the vehicle) and both ends' transmit power: those live in the
ELRS link statistics, which the MAVLink link mode does not forward (see
`docs/flight-data-interpretation.md`, per-antenna RSSI).

In [ ]:
# --- papermill parameters ---
input_file = "/home/jovyan/datasets/flights/rekon10/2026-10-27-ffocus/fc-log-71.bin"
flight_dir = ""   # "" -> the .bin's directory
out_dir = ""      # "" -> <flight_dir>/derived
max_gap_s = 2.0   # a layer value is placed on a POS point only if one exists within this

In [ ]:
import json, os, sys, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
from IPython.display import display

REPO = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "analysis"))
from ardupilot_log import parse_log
import vehicle_health as vh

LOG = Path(input_file)
FLIGHT = Path(flight_dir) if flight_dir else LOG.parent
OUT = Path(out_dir) if out_dir else FLIGHT / "derived"
OUT.mkdir(parents=True, exist_ok=True)
STEM = LOG.stem.replace(" ", "-")

F, duration, parms = parse_log(LOG, {"POS", "GPS", "GPA", "RAD", "UBX1", "MAV", "EV"})
print(f"{LOG.name}: {duration:.0f} s;", ", ".join(f"{k} {len(v)}" for k, v in sorted(F.items())))
pos = F.get("POS", pd.DataFrame())
assert not pos.empty, "no POS rows: nothing to place layers on"
FIT = vh.fc_utc_fit(F.get("GPS"))
print("FC->UTC fit:", "none (no dated GPS rows; ground and coordinator layers are skipped)" if FIT is None
      else f"slope {FIT[0]:.9e}")

# Armed spans from EV (10 armed, 11 disarmed), on TimeUS.
ev = F.get("EV", pd.DataFrame())
ARMED = []
if not ev.empty:
    t_arm = None
    for _, r in ev.sort_values("TimeUS").iterrows():
        if r["Id"] == 10 and t_arm is None: t_arm = r["TimeUS"]
        elif r["Id"] == 11 and t_arm is not None: ARMED.append((t_arm, r["TimeUS"])); t_arm = None
    if t_arm is not None: ARMED.append((t_arm, pos["TimeUS"].max()))
print("armed spans (s from log start):",
      [(round((a - pos.TimeUS.min()) / 1e6), round((b - pos.TimeUS.min()) / 1e6)) for a, b in ARMED])

def in_armed(tus):
    m = np.zeros(len(tus), bool)
    for a, b in ARMED: m |= (tus >= a) & (tus <= b)
    return m

## Layers

Each layer is a table of `TimeUS` and a value. FC streams carry `TimeUS` already; the
coordinator's collectd and the ground metrics are unix time, put on `TimeUS` through the
FC's GPS fit.

In [ ]:
LAYERS = {}   # name -> dict(df[TimeUS,value], units, source)

def add(name, tus, values, units, source):
    df = pd.DataFrame({"TimeUS": np.asarray(tus, float), "value": np.asarray(values, float)}).dropna()
    if df.empty:
        print(f"  {name}: no rows"); return
    LAYERS[name] = dict(df=df.sort_values("TimeUS"), units=units, source=source)

def utc_to_tus(t):
    return (np.asarray(t, float) - FIT[1]) / FIT[0]

g = F.get("GPS", pd.DataFrame())
if not g.empty:
    g = g[g["I"] == 0]
    add("GPS fix type", g.TimeUS, g.Status, "0-6 (3 3D, 4 DGPS, 5 RTK float, 6 RTK fixed)", "GPS.Status")
    add("GPS satellites", g.TimeUS, g.NSats, "count", "GPS.NSats")
    add("GPS HDop", g.TimeUS, g.HDop, "", "GPS.HDop")
a = F.get("GPA", pd.DataFrame())
if not a.empty:
    a = a[a["I"] == 0]
    add("GPS HAcc", a.TimeUS, a.HAcc, "m", "GPA.HAcc")
    add("GPS VAcc", a.TimeUS, a.VAcc, "m", "GPA.VAcc")
u = F.get("UBX1", pd.DataFrame())
if not u.empty:
    add("F9P jamInd", u.TimeUS, u.jamInd, "0-255", "UBX1.jamInd")
    add("F9P noisePerMS", u.TimeUS, u.noisePerMS, "", "UBX1.noisePerMS")
    add("F9P agcCnt", u.TimeUS, u.agcCnt, "0-8191", "UBX1.agcCnt")
r = F.get("RAD", pd.DataFrame())
if not r.empty:
    e = vh.elrs_radio(r)
    add("ELRS uplink LQ", e.TimeUS, e.uplink_lq_pct, "%", "RAD.RSSI / 2.55")
    add("ELRS uplink RSSI radio 1", e.TimeUS, e.uplink_rssi1_dbm, "dBm", "-RAD.RemRSSI")
    add("ELRS uplink SNR", e.TimeUS, e.uplink_snr_db, "dB", "RAD.Noise (int8)")
    add("ELRS receiver MAVLink buffer free", e.TimeUS, e.rx_mavlink_buf_free_pct, "%", "RAD.TxBuf")
mv = F.get("MAV", pd.DataFrame())
if not mv.empty:
    ch = mv[mv["chan"] == 2].copy()
    ch["t"] = ch["TimeUS"] / 1e6
    for col, label in (("rxp", "rx"), ("txp", "tx"), ("rxdp", "rx dropped")):
        rr = vh.counter_rate(ch, col)
        add(f"ELRS link MAVLink {label}", rr.t * 1e6, rr.rate, "packets/s", f"MAV chan 2 {col}, differenced")

if FIT is not None:
    cd = sorted((FLIGHT / "captures" / "coordinator").glob("*/collectd/coordinator"))
    t0, t1 = (pos.TimeUS.min() * FIT[0] + FIT[1]), (pos.TimeUS.max() * FIT[0] + FIT[1])
    for d in cd:
        s = vh.collectd_series(d)
        s = s[(s.plugin == "interface-wlan0") & (s.t >= t0 - 60) & (s.t <= t1 + 60)]
        if s.empty: continue
        print(f"coordinator collectd {d.parent.parent.name}: wlan0 rows in this log's span")
        for typ, col, label, units in (("if_octets", "rx", "rx", "bytes/s"), ("if_octets", "tx", "tx", "bytes/s"),
                                       ("if_dropped", "rx", "rx dropped", "packets/s"),
                                       ("if_errors", "rx", "rx errors", "packets/s"),
                                       ("if_errors", "tx", "tx errors", "packets/s")):
            ss = s[(s.type == typ) & (s.column == col)]
            if ss.empty: continue
            rr = vh.counter_rate(ss, "value")
            add(f"coordinator wlan0 {label}", utc_to_tus(rr.t), rr.rate, units, f"collectd {typ}.{col}, differenced")
    for f in sorted((FLIGHT / "ground").glob("backpack-metrics-*.json")):
        res = json.loads(f.read_text()).get("data", {}).get("result", [])
        for series in res:
            name = series["metric"].get("__name__")
            if name not in ("backpack_mavlink_packets_down_total", "backpack_mavlink_drops_down_total"): continue
            v = pd.DataFrame(series["values"], columns=["t", "value"]).astype(float)
            v = v[(v.t >= t0 - 60) & (v.t <= t1 + 60)]
            if v.empty: continue
            rr = vh.counter_rate(v, "value")
            label = "packets down" if "packets" in name else "drops down"
            add(f"ground backpack MAVLink {label}", utc_to_tus(rr.t), rr.rate, "per s", f"{f.name} {name}, differenced")

print(f"{len(LAYERS)} layers:", ", ".join(LAYERS))

## Placed on the track

`POS` is the EKF's position. A layer value goes on a `POS` point only if the layer has a
sample within `max_gap_s` of it (sparse streams like `UBX1`, every few seconds, get a wider
window of five times their median interval) -- a gap in a layer shows as a gap on the map.

In [ ]:
track = pos[["TimeUS", "Lat", "Lng", "RelHomeAlt"]].sort_values("TimeUS").reset_index(drop=True)
track["TimeUS"] = track["TimeUS"].astype(float)
track["armed"] = in_armed(track.TimeUS.values)
PLACED = {}
for name, L in LAYERS.items():
    df = L["df"]
    med = float(np.median(np.diff(df.TimeUS))) / 1e6 if len(df) > 2 else max_gap_s
    tol = max(max_gap_s, 5 * med)
    m = pd.merge_asof(track, df, on="TimeUS", direction="nearest", tolerance=tol * 1e6)
    PLACED[name] = m.dropna(subset=["value"])
    L["tolerance_s"] = tol

def ll_scatter(ax, m, name, units):
    k = np.cos(np.radians(track.Lat.mean()))
    ax.plot(track.Lng, track.Lat, "-", color="0.85", lw=0.8, zorder=0)
    sc = ax.scatter(m.Lng, m.Lat, c=m.value, s=4, cmap="viridis", zorder=1)
    ax.set_aspect(1 / k); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}" + (f" ({units})" if units else ""), fontsize=8)
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.02).ax.tick_params(labelsize=7)

names = list(PLACED)
ncol = 3; nrow = int(np.ceil(len(names) / ncol)) or 1
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 3.6 * nrow))
for ax in np.ravel(axes)[len(names):]: ax.axis("off")
for ax, name in zip(np.ravel(axes), names):
    m = PLACED[name]
    m = m[m.armed] if m.armed.any() else m
    ll_scatter(ax, m, name, LAYERS[name]["units"])
fig.suptitle(f"{LOG.name}: layers on the EKF track" + (" (armed only)" if ARMED else ""), fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.98)); fig.savefig(OUT / f"link-map-{STEM}.png", dpi=110); plt.show()

## Over time

The same layers against time, armed spans shaded, for what the map cannot show: when
something changed, and whether two layers moved together.

In [ ]:
t0 = track.TimeUS.min()
fig, axes = plt.subplots(len(names), 1, figsize=(12, 1.4 * len(names)), sharex=True)
for ax, name in zip(np.atleast_1d(axes), names):
    df = LAYERS[name]["df"]
    ax.plot((df.TimeUS - t0) / 1e6, df.value, ".", ms=1.5)
    for a, b in ARMED: ax.axvspan((a - t0) / 1e6, (b - t0) / 1e6, color="tab:orange", alpha=0.12, lw=0)
    ax.set_ylabel(name, rotation=0, ha="right", fontsize=7); ax.tick_params(labelsize=7)
np.atleast_1d(axes)[-1].set_xlabel("s from log start")
fig.tight_layout(); fig.savefig(OUT / f"link-map-{STEM}-time.png", dpi=110); plt.show()

## Interactive map

One layer per stream, switchable, with the value on hover. Written to
`derived/link-map-<log>.html`; open it in a browser (the tiles need network).

In [ ]:
fmap = folium.Map(location=[track.Lat.mean(), track.Lng.mean()], zoom_start=18, max_zoom=22, tiles=None)
folium.TileLayer("OpenStreetMap", max_zoom=22, max_native_zoom=19).add_to(fmap)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri World Imagery", name="Imagery", max_zoom=22, max_native_zoom=19).add_to(fmap)
folium.PolyLine(track[["Lat", "Lng"]].values.tolist(), color="#888", weight=1, name="EKF track").add_to(fmap)
for i, name in enumerate(names):
    m = PLACED[name]
    if m.empty: continue
    m = m.iloc[:: max(1, len(m) // 1500)]
    lo, hi = float(m.value.min()), float(m.value.max())
    cmap = cm.LinearColormap(["#440154", "#21918c", "#fde725"], vmin=lo, vmax=hi if hi > lo else lo + 1)
    units = LAYERS[name]["units"]
    fg = folium.FeatureGroup(name=f"{name} [{lo:.4g} .. {hi:.4g}] {units}", show=(i == 0))
    for _, p in m.iterrows():
        folium.CircleMarker([p.Lat, p.Lng], radius=3, color=cmap(p.value), fill=True, fill_opacity=0.9,
                            weight=0, tooltip=f"{name}: {p.value:.4g} {units}").add_to(fg)
    fg.add_to(fmap)
folium.LayerControl(collapsed=False).add_to(fmap)
html = OUT / f"link-map-{STEM}.html"
fmap.save(str(html))
print("wrote", html)

## Summary

Per layer: the number of samples and their range, whole log and armed only, and the share of
armed track points that have a value. No thresholds and no verdicts -- the figures are for
reading and for comparing against later flights.

In [ ]:
rows = []; REPORT = dict(notebook="link-map.ipynb", input_file=str(LOG), layers={})
for name, L in LAYERS.items():
    df = L["df"]; m = PLACED[name]
    arm = df[in_armed(df.TimeUS.values)] if ARMED else df.iloc[0:0]
    n_track_armed = int(track.armed.sum())
    cov = (int(m.armed.sum()) / n_track_armed) if n_track_armed else None
    REPORT["layers"][name] = dict(source=L["source"], units=L["units"], tolerance_s=L["tolerance_s"],
                                  whole=vh.summarize(df.value), armed=vh.summarize(arm.value),
                                  armed_track_coverage=cov)
    s = REPORT["layers"][name]["armed"] if ARMED else REPORT["layers"][name]["whole"]
    rows.append(dict(layer=name, units=L["units"], n=s.get("n"), min=s.get("min"), median=s.get("median"),
                     max=s.get("max"), armed_coverage=None if cov is None else round(cov, 3)))
REPORT["not_recorded"] = ["coordinator wifi RSSI (no collectd wireless plugin)",
                          "ELRS radio 2 RSSI, active radio, downlink RSSI/LQ/SNR, transmit power (not in RADIO_STATUS)"]
display(pd.DataFrame(rows).set_index("layer"))
(OUT / f"link-map-{STEM}.json").write_text(json.dumps(REPORT, indent=2, default=float) + "\n")
print("wrote", OUT / f"link-map-{STEM}.json")